In [2]:
# import requests
# import json
# import csv
# import time
# import os
# # ── CONFIG ──────────────────────────────────────────────────────────────────
# BASE_URL   = "https://keyvalue.hamropatro.com/kv/get"
# ELECTION   = "2082"
# OUTPUT_CSV  = "nepal_election_2082_results.csv"
# OUTPUT_JSON = "nepal_election_2082_results.json"
# DELAY       = 0.3   # seconds between requests (be polite to the server)
# # ────────────────────────────────────────────────────────────────────────────
# def fetch_districts():
#     """Fetch the list of all 77 districts with their IDs."""
#     url = f"{BASE_URL}/major-election-{ELECTION}-districts"
#     resp = requests.get(url, timeout=10)
#     resp.raise_for_status()
#     raw = resp.json()
#     return json.loads(raw["list"][0]["value"])
# def fetch_district_results(district_id):
#     """Fetch candidate results for a single district."""
#     url = f"{BASE_URL}/major-election-{ELECTION}-districts-{district_id}-result::-1"
#     resp = requests.get(url, timeout=10)
#     resp.raise_for_status()
#     raw = resp.json()
#     value = raw.get("list", [{}])[0].get("value", "")
#     if not value:
#         return None
#     return json.loads(value)
# # ── MAIN SCRAPE ──────────────────────────────────────────────────────────────
# print("Fetching district list...")
# districts = fetch_districts()
# print(f"  Found {len(districts)} districts.\\n")
# # Build a lookup: district_id -> stateId (province)
# district_meta = {d["key"]: d for d in districts}
# all_rows = []
# failed   = []
# for i, district in enumerate(districts, 1):
#     district_id   = district["key"]
#     district_name = district["englishName"]
#     state_id      = district["stateId"]
#     print(f"[{i:02d}/77] {district_name} (ID: {district_id}) ...", end=" ")
#     try:
#         result = fetch_district_results(district_id)
#         if result is None:
#             print("No data.")
#             failed.append(district_name)
#             continue
#         area_results = result.get("districtResults", [])
#         candidate_count = sum(len(a["candidateResults"]) for a in area_results)
#         print(f"{len(area_results)} constituencies, {candidate_count} candidates")
#         for area in area_results:
#             for candidate in area["candidateResults"]:
#                 row = {
#                     # Election metadata
#                     "election_id":          ELECTION,
#                     "election_type":        area.get("type", "HOR"),  # HOR = House of Representatives
#                     # Province / District
#                     "state_id":             state_id,
#                     "district_id":          district_id,
#                     "district_name_np":     area.get("districtName", ""),
#                     "district_name_en":     area.get("districtEnglishName", ""),
#                     # Constituency
#                     "area_id":              area.get("areaId", ""),
#                     "area_name_np":         area.get("areaName", ""),
#                     "area_name_en":         area.get("areaNameEnglish", ""),
#                     "registered_voters":    area.get("registeredVoters", ""),
#                     "previous_voters":      area.get("previousVoters", ""),
#                     "voters_increased":     area.get("votersIncreased", ""),
#                     "result_final":         area.get("resultFinal", ""),
#                     # Candidate
#                     "candidate_id":         candidate.get("candidateId", ""),
#                     "candidate_name_np":    candidate.get("name", ""),
#                     "candidate_name_en":    candidate.get("englishName", ""),
#                     "candidate_slug":       candidate.get("slug", ""),
#                     # Party
#                     "party_id":             candidate.get("partyId", ""),
#                     "party_name_np":        candidate.get("partyName", ""),
#                     # Results
#                     "votes":                candidate.get("votes", 0),
#                     "winner":               candidate.get("winner", False),
#                     "leading":              candidate.get("leading", False),
#                     # Image
#                     "image_url":            candidate.get("imageUrl", ""),
#                 }
#                 all_rows.append(row)
#     except Exception as e:
#         print(f"ERROR: {e}")
#         failed.append(district_name)
#     time.sleep(DELAY)
# # ── SAVE OUTPUTS ─────────────────────────────────────────────────────────────
# print(f"\\n{'─'*60}")
# print(f"Total candidate rows collected: {len(all_rows)}")
# # Save JSON
# with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
#     json.dump(all_rows, f, ensure_ascii=False, indent=2)
# print(f"Saved JSON  → {os.path.abspath(OUTPUT_JSON)}")
# # Save CSV
# if all_rows:
#     with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as f:  # utf-8-sig for Excel compatibility
#         writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
#         writer.writeheader()
#         writer.writerows(all_rows)
#     print(f"Saved CSV   → {os.path.abspath(OUTPUT_CSV)}")
# if failed:
#     print(f"\\nFailed districts ({len(failed)}): {', '.join(failed)}")
# else:
#     print("\\nAll districts fetched successfully!")

In [3]:
import requests
import json
import csv
import time
from pathlib import Path
# ── CONFIG ──────────────────────────────────────────────────────────────────
BASE_URL    = "https://keyvalue.hamropatro.com/kv/get"
ELECTION    = "2082"
OUTPUT_CSV  = "nepal_election_2082_results.csv"
OUTPUT_JSON = "nepal_election_2082_results.json"
CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "Codes" else CURRENT_DIR
OUTPUT_DIR  = PROJECT_DIR / "Exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV
OUTPUT_JSON_PATH = OUTPUT_DIR / OUTPUT_JSON
DELAY       = 0.3   # seconds between requests
# ────────────────────────────────────────────────────────────────────────────
def fetch_all_areas():
    """Fetch the list of all 165 constituencies with metadata."""
    url = f"{BASE_URL}/major-election-{ELECTION}-areas"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    raw = resp.json()
    return json.loads(raw["list"][0]["value"])
def fetch_all_districts():
    """Fetch district name lookup (id -> name)."""
    url = f"{BASE_URL}/major-election-{ELECTION}-districts"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    raw = resp.json()
    districts = json.loads(raw["list"][0]["value"])
    return {d["key"]: d["englishName"] for d in districts}
def fetch_area_results(area_id):
    """Fetch ALL candidate results for a single constituency."""
    # The key format bundles 2074 + 2079 + 2082; we request all three and pick 2082
    key = (
        f"major-election-2074-areas-{area_id}-result::-1"
        f"::major-election-2079-areas-{area_id}-result::-1"
        f"::major-election-{ELECTION}-areas-{area_id}-result::-1"
    )
    url = f"{BASE_URL}/{key}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    raw = resp.json()
    # Find the 2082 entry in the returned list
    entry = next((item for item in raw.get("list", [])
                  if f"election-{ELECTION}" in item["key"] and item.get("value")), None)
    if not entry:
        return None
    return json.loads(entry["value"])
# ── MAIN SCRAPE ──────────────────────────────────────────────────────────────
print("Fetching areas list...")
areas = fetch_all_areas()
print(f"  Found {len(areas)} constituencies.\\n")
print("Fetching districts lookup...")
district_names = fetch_all_districts()
print(f"  Found {len(district_names)} districts.\\n")
all_rows = []
failed   = []
for i, area in enumerate(sorted(areas, key=lambda a: int(a["id"])), 1):
    area_id   = area["id"]
    area_name = area["englishName"]
    dist_id   = area["districtId"]
    dist_name = district_names.get(dist_id, f"District {dist_id}")
    print(f"[{i:03d}/165] {dist_name} › {area_name} (Area ID: {area_id}) ...", end=" ")
    try:
        result = fetch_area_results(area_id)
        if result is None:
            print("No data.")
            failed.append(f"{dist_name} - {area_name}")
            continue
        candidates = result.get("candidateResults", [])
        print(f"{len(candidates)} candidates")
        for candidate in candidates:
            row = {
                # Election
                "election_id":            ELECTION,
                "election_type":          result.get("type", "HOR"),
                # Province
                "state_id":               result.get("stateId", area.get("stateId", "")),
                "state_name":             result.get("stateName", ""),
                # District
                "district_id":            dist_id,
                "district_name_en":       result.get("districtEnglishName", dist_name),
                "district_name_np":       result.get("districtName", ""),
                # Constituency
                "area_id":                area_id,
                "area_name_en":           result.get("areaNameEnglish", area_name),
                "area_name_np":           result.get("areaName", area.get("name", "")),
                "registered_voters":      result.get("registeredVoters", area.get("registeredVoters", "")),
                "previous_voters":        result.get("previousVoters", area.get("previousVoters", "")),
                "voters_increased":       result.get("votersIncreased", area.get("votersIncreased", "")),
                "total_counted_votes":    result.get("totalCountedVotes", ""),
                "total_cast_votes":       area.get("totalCastVotes", ""),
                "result_status":          result.get("electionResultStatus", ""),
                "result_final":           result.get("resultFinal", ""),
                # Candidate
                "candidate_id":           candidate.get("candidateId", ""),
                "candidate_name_en":      candidate.get("englishName", ""),
                "candidate_name_np":      candidate.get("name", ""),
                "candidate_slug":         candidate.get("slug", ""),
                "candidate_image_url":    candidate.get("imageUrl", ""),
                # Party
                "party_id":               candidate.get("partyId", ""),
                "party_name_np":          candidate.get("partyName", ""),
                # Results
                "votes":                  candidate.get("votes", 0),
                "winner":                 candidate.get("winner", False),
                "leading":                candidate.get("leading", False),
            }
            all_rows.append(row)
    except Exception as e:
        print(f"ERROR: {e}")
        failed.append(f"{dist_name} - {area_name}")
    time.sleep(DELAY)
# ── SAVE OUTPUTS ─────────────────────────────────────────────────────────────
print(f"\\n{'─'*60}")
print(f"Total candidate rows: {len(all_rows)}")
# Save JSON
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)
print(f"Saved JSON  → {OUTPUT_JSON_PATH}")
# Save CSV (utf-8-sig for Excel compatibility with Nepali text)
if all_rows:
    with open(OUTPUT_CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"Saved CSV   → {OUTPUT_CSV_PATH}")
if failed:
    print(f"\\nFailed ({len(failed)}): {', '.join(failed)}")
else:
    print("\\nAll 165 constituencies fetched successfully!")

Fetching areas list...
  Found 165 constituencies.\n
Fetching districts lookup...
  Found 77 districts.\n
[001/165] Taplejung › Area 1 (Area ID: 1) ... 13 candidates
[002/165] Panchthar › Area 1 (Area ID: 2) ... 15 candidates
[003/165] Ilam › Area 1 (Area ID: 3) ... 13 candidates
[004/165] Ilam › Area 2 (Area ID: 4) ... 14 candidates
[005/165] Jhapa › Area 1 (Area ID: 5) ... 15 candidates
[006/165] Jhapa › Area 2 (Area ID: 6) ... 18 candidates
[007/165] Jhapa › Area 3 (Area ID: 7) ... 17 candidates
[008/165] Jhapa › Area 4 (Area ID: 8) ... 16 candidates
[009/165] Jhapa › Area 5 (Area ID: 9) ... 24 candidates
[010/165] Sankhuwasabha › Area 1 (Area ID: 10) ... 16 candidates
[011/165] Terhathum › Area 1 (Area ID: 11) ... 12 candidates
[012/165] Bhojpur › Area 1 (Area ID: 12) ... 12 candidates
[013/165] Dhankuta › Area 1 (Area ID: 13) ... 17 candidates
[014/165] Morang › Area 1 (Area ID: 14) ... 17 candidates
[015/165] Morang › Area 2 (Area ID: 15) ... 17 candidates
[016/165] Morang › Area

In [4]:
import requests
import json
import csv
import time
from pathlib import Path

# ── CONFIG (must match previous cell) ────────────────────────────────────────
BASE_URL    = "https://keyvalue.hamropatro.com/kv/get"
ELECTION    = "2082"
OUTPUT_CSV  = "nepal_election_2082_results.csv"
OUTPUT_JSON = "nepal_election_2082_results.json"
CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "Codes" else CURRENT_DIR
OUTPUT_DIR  = PROJECT_DIR / "Exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = OUTPUT_DIR / OUTPUT_CSV
OUTPUT_JSON_PATH = OUTPUT_DIR / OUTPUT_JSON
DELAY       = 0.5   # slightly longer delay for retries
# ─────────────────────────────────────────────────────────────────────────────

# ── LOAD EXISTING DATA ────────────────────────────────────────────────────────
print("Loading existing exported data...")
with open(OUTPUT_JSON_PATH, encoding="utf-8") as f:
    all_rows = json.load(f)

existing_area_ids = {str(row["area_id"]) for row in all_rows}
print(f"  Loaded {len(all_rows)} rows covering {len(existing_area_ids)} area IDs.")

# ── IDENTIFY MISSING AREAS ────────────────────────────────────────────────────
print("\nFetching area and district metadata to identify missing areas...")

def fetch_all_areas_retry():
    url = f"{BASE_URL}/major-election-{ELECTION}-areas"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    return json.loads(resp.json()["list"][0]["value"])

def fetch_all_districts_retry():
    url = f"{BASE_URL}/major-election-{ELECTION}-districts"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    districts = json.loads(resp.json()["list"][0]["value"])
    return {d["key"]: d["englishName"] for d in districts}

def fetch_area_results_retry(area_id):
    key = (
        f"major-election-2074-areas-{area_id}-result::-1"
        f"::major-election-2079-areas-{area_id}-result::-1"
        f"::major-election-{ELECTION}-areas-{area_id}-result::-1"
    )
    url = f"{BASE_URL}/{key}"
    resp = requests.get(url, timeout=20)   # longer timeout for retries
    resp.raise_for_status()
    raw = resp.json()
    entry = next(
        (item for item in raw.get("list", [])
         if f"election-{ELECTION}" in item["key"] and item.get("value")),
        None,
    )
    if not entry:
        return None
    return json.loads(entry["value"])

areas         = fetch_all_areas_retry()
district_names = fetch_all_districts_retry()
print(f"  Found {len(areas)} areas and {len(district_names)} districts.")

# Filter to only areas not present in the exported file
missing_areas = [a for a in areas if str(a["id"]) not in existing_area_ids]
print(f"\nMissing area IDs: {[a['id'] for a in missing_areas]}")

if not missing_areas:
    print("\nNo missing areas — nothing to retry!")
else:
    # ── RETRY MISSING AREAS ───────────────────────────────────────────────────
    new_rows    = []
    still_failed = []

    for area in sorted(missing_areas, key=lambda a: int(a["id"])):
        area_id   = area["id"]
        area_name = area["englishName"]
        dist_id   = area["districtId"]
        dist_name = district_names.get(dist_id, f"District {dist_id}")
        print(f"Retrying: {dist_name} › {area_name} (Area ID: {area_id}) ...", end=" ")

        try:
            result = fetch_area_results_retry(area_id)
            if result is None:
                print("No data.")
                still_failed.append(f"{dist_name} - {area_name}")
                continue
            candidates = result.get("candidateResults", [])
            print(f"{len(candidates)} candidates")
            for candidate in candidates:
                row = {
                    "election_id":         ELECTION,
                    "election_type":       result.get("type", "HOR"),
                    "state_id":            result.get("stateId", area.get("stateId", "")),
                    "state_name":          result.get("stateName", ""),
                    "district_id":         dist_id,
                    "district_name_en":    result.get("districtEnglishName", dist_name),
                    "district_name_np":    result.get("districtName", ""),
                    "area_id":             area_id,
                    "area_name_en":        result.get("areaNameEnglish", area_name),
                    "area_name_np":        result.get("areaName", area.get("name", "")),
                    "registered_voters":   result.get("registeredVoters", area.get("registeredVoters", "")),
                    "previous_voters":     result.get("previousVoters", area.get("previousVoters", "")),
                    "voters_increased":    result.get("votersIncreased", area.get("votersIncreased", "")),
                    "total_counted_votes": result.get("totalCountedVotes", ""),
                    "total_cast_votes":    area.get("totalCastVotes", ""),
                    "result_status":       result.get("electionResultStatus", ""),
                    "result_final":        result.get("resultFinal", ""),
                    "candidate_id":        candidate.get("candidateId", ""),
                    "candidate_name_en":   candidate.get("englishName", ""),
                    "candidate_name_np":   candidate.get("name", ""),
                    "candidate_slug":      candidate.get("slug", ""),
                    "candidate_image_url": candidate.get("imageUrl", ""),
                    "party_id":            candidate.get("partyId", ""),
                    "party_name_np":       candidate.get("partyName", ""),
                    "votes":               candidate.get("votes", 0),
                    "winner":              candidate.get("winner", False),
                    "leading":             candidate.get("leading", False),
                }
                new_rows.append(row)
        except Exception as e:
            print(f"ERROR: {e}")
            still_failed.append(f"{dist_name} - {area_name}")
        time.sleep(DELAY)

    # ── MERGE, SORT BY area_id, AND SAVE ─────────────────────────────────────
    if new_rows:
        combined = all_rows + new_rows
        combined.sort(key=lambda r: int(r["area_id"]))

        with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(combined, f, ensure_ascii=False, indent=2)
        print(f"\nSaved JSON  → {OUTPUT_JSON_PATH}")

        with open(OUTPUT_CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=combined[0].keys())
            writer.writeheader()
            writer.writerows(combined)
        print(f"Saved CSV   → {OUTPUT_CSV_PATH}")

        recovered = len(missing_areas) - len(still_failed)
        print(f"\nAdded {len(new_rows)} rows from {recovered} recovered area(s).")
        print(f"Total rows now: {len(combined)}")
    else:
        print("\nNo new rows were retrieved.")

    if still_failed:
        print(f"\nStill failed ({len(still_failed)}): {', '.join(still_failed)}")
    else:
        print("\nAll previously missing areas recovered successfully!")


Loading existing exported data...
  Loaded 3406 rows covering 165 area IDs.

Fetching area and district metadata to identify missing areas...
  Found 165 areas and 77 districts.

Missing area IDs: []

No missing areas — nothing to retry!
